In [1]:
%load_ext cudf.pandas
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import seaborn as sns 
import matplotlib.pyplot as plt 
import time
import cupy as cp

In [2]:
data = pd.read_csv('/home/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/input/nfl-big-data-bowl-2023/week1.csv')
scout = pd.read_csv('/home/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/input/nfl-big-data-bowl-2023/pffScoutingData.csv')
plays = pd.read_csv('/home/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/input/nfl-big-data-bowl-2023/plays.csv')
players = pd.read_csv('/home/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/input/nfl-big-data-bowl-2023/players.csv')


# Let's merge these data into one set 
data = data.merge(scout, how='left')

# total_bytes = (
#     data.memory_usage(deep=True).sum()
#     + scout.memory_usage(deep=True).sum()
#     + plays.memory_usage(deep=True).sum()
#     + players.memory_usage(deep=True).sum()
# )

# # Convert to megabytes and print
# total_mb = total_bytes / (1024 ** 2)
# print(f"Total memory usage: {total_mb:.2f} MB")

In [3]:
data = pd.concat([data]*factor)
# total_bytes = (
#     data.memory_usage(deep=True).sum()
#     + scout.memory_usage(deep=True).sum()
#     + plays.memory_usage(deep=True).sum()
#     + players.memory_usage(deep=True).sum()
# )

# # Convert to megabytes and print
# total_mb = total_bytes / (1024 ** 2)
# print(f"Total memory usage: {total_mb:.2f} MB")

In [4]:
### cell 0
# get ball snap indicies 
_idxs = (data
         .loc[data['event']=='ball_snap', 
              'frameId']
         .index
         .values)

# to get 500ms of movement after snap, get 5 rows (each row is 100ms of info)
x = [(_idxs+x).tolist() for x in range(0,6)]
idxs = [item for sublist in x for item in sublist] #the output x is a list of lists, so this is just to flatten the list

# filter for snap + 500ms of data only using our selected indicies
_df = data.loc[idxs]

# total_bytes = (
#     data.memory_usage(deep=True).sum()
#     + scout.memory_usage(deep=True).sum()
#     + plays.memory_usage(deep=True).sum()
#     + players.memory_usage(deep=True).sum()
#     + _df.memory_usage(deep=True).sum()
# )

# # Convert to megabytes and print
# total_mb = total_bytes / (1024 ** 2)
# print(f"Total memory usage: {total_mb:.2f} MB")

In [5]:
### cell 1
gid = 2021090900
pid = 97 
nid = 25511 
_ = _df.loc[(_df['gameId']==gid) & (_df['playId']==pid) & (_df['nflId']==nid)]

In [6]:
%%time
### cell 2
_los = (data
        .loc[(data['team']=='football') & 
             (data['frameId']==1), 
             ['gameId', 'playId', 'x']]
        .rename(columns={'x':'los'}))

# merge LOS info back to subsetted data
_df = _df.merge(_los)

# total_bytes = (
#     data.memory_usage(deep=True).sum()
#     + scout.memory_usage(deep=True).sum()
#     + plays.memory_usage(deep=True).sum()
#     + players.memory_usage(deep=True).sum()
#     + _df.memory_usage(deep=True).sum()
# )

# # Convert to megabytes and print
# total_mb = total_bytes / (1024 ** 2)
# print(f"Total memory usage: {total_mb:.2f} MB")

CPU times: user 140 ms, sys: 58.8 ms, total: 198 ms
Wall time: 150 ms


Generating '/var/tmp/nsys-report-f003.qdstrm'
[1/2] [========================100%] cell_2_2025-05-21T17:28:24.nsys-rep
[2/2] [========================100%] cell_2_2025-05-21T17:28:24.sqlite
Generated:
	/home/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/src/plot_curves/cell_2_2025-05-21T17:28:24.nsys-rep
	/home/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/src/plot_curves/cell_2_2025-05-21T17:28:24.sqlite
Generating SQLite file /home/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/src/plot_curves/cell_2_2025-05-21T17:28:24.sqlite from /home/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/src/plot_curves/cell_2_2025-05-21T17:28:24.nsys-rep
Processing [/home/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/src/plot_curves/cell_2_2025-05-21T17:28:24.sqlite] with [/opt/nvidia/nsight-systems/2025.3.1/host-linux-x64/reports/nvtx_sum.py]... 

 ** NVTX 